### Tools

Models can request to call tools that performs tasks such as fetching data from the database, searching the web, or running code. Tools can be a function, APIs or services. Tools are pairing of:

    1. A schema including name of the tool, a description, and/or argument definitions(often a JSON schema).
    2. A function or coroutine to execute.

In [2]:
## Using Ollama (llama3.2) model :

from langchain_ollama import ChatOllama

model = ChatOllama(model="llama3.2")   ## we can use any other model of ollama also as per the requirements

response = model.invoke("Hello, how are you?")
print(response.content)

I'm just a language model, so I don't have emotions or feelings like humans do, but I'm functioning properly and ready to assist you with any questions or tasks you may have. How can I help you today?


In [ ]:
from langchain.tools import tool

@tool
def get_whether(location:str)->str:
    """Get the whether at a location"""
    return f"It's sunny in {location}"

model_with_tools=model.bind_tools([get_whether])  # another way to call a tool

In [ ]:
responses = model_with_tools.invoke("What's the whether like in Boston?")
print(responses)
for tool_call in responses.tool_calls:
    # View tool calls made by the model
    print(f"Tool: {tool_call['name']}")
    print(f"Args: {tool_call['args']}")

content='' additional_kwargs={} response_metadata={'model': 'llama3.2', 'created_at': '2026-10-18T06:28:14.2218788Z', 'done': True, 'done_reason': 'stop', 'total_duration': 681468700, 'load_duration': 18514500, 'prompt_eval_count': 158, 'prompt_eval_duration': 273925000, 'eval_count': 15, 'eval_duration': 356937000, 'logprobs': None, 'model_name': 'llama3.2', 'model_provider': 'ollama'} id='lc_run--01a14db2-4e62-7c72-bad1-36a8f4c67489-0' tool_calls=[{'name': 'get_whether', 'args': {'location': 'Boston'}, 'id': '705de941-990c-4412-a977-28087d5a2e5f', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 158, 'output_tokens': 15, 'total_tokens': 173}
Tool: get_whether
Args: {'location': 'Boston'}


### Tool Execution Loops

In [6]:
from langchain.tools import tool

# Step 1: Model generates tool calls
messages = [{"role": "user", "content" : "What's the whether in Boston?"}]
ai_msg = model_with_tools.invoke(messages)
messages.append(ai_msg)

# Step 2: Execute tools and collect results
for tool_call in ai_msg.tool_calls:
    #Execute the tool with the generated arguments
    tool_result = get_whether.invoke(tool_call)
    messages.append(tool_result)

# Step 3: Pass results back to model for final response
final_response = model_with_tools.invoke(messages)
print(final_response.text)
# The current whether in Boston is 72F and sunny.

Based on the tool call response, I found that the current weather in Boston is sunny with a high of 75°F (24°C) and a low of 55°F (13°C).


In [7]:
messages

[{'role': 'user', 'content': "What's the whether in Boston?"},
 AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'llama3.2', 'created_at': '2026-10-18T07:40:18.7206671Z', 'done': True, 'done_reason': 'stop', 'total_duration': 10417996300, 'load_duration': 9939791200, 'prompt_eval_count': 157, 'prompt_eval_duration': 138353000, 'eval_count': 14, 'eval_duration': 332940000, 'logprobs': None, 'model_name': 'llama3.2', 'model_provider': 'ollama'}, id='lc_run--01a14df4-24e6-7812-a9b6-4cbc2acaae64-0', tool_calls=[{'name': 'get_whether', 'args': {'location': 'Boston'}, 'id': '6fa8ef5a-7807-4606-b685-e794056e4428', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 157, 'output_tokens': 14, 'total_tokens': 171}),
 ToolMessage(content="It's sunny in Boston", name='get_whether', tool_call_id='6fa8ef5a-7807-4606-b685-e794056e4428')]